# Pilote EdS — à exécuter dans Google Colab

Pilote sur **3 dossiers** du canevas anonymisé (EDS-04 pour le calibrage, EDS-02 et EDS-09 pour la mesure) : le premier sert au calibrage du seuil d'appariement, les deux
autres à la mesure. 3 dossiers × 3 modèles × 2 conditions × 5 répétitions = **90 appels**.

**Avant d'exécuter :**
- autorisation écrite d'exploitation des dossiers obtenue ;
- canevas **anonymisé** (identifiants EDS-xx, aucun nom de centre ni de participant) ;
- clés `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GOOGLE_API_KEY` dans le gestionnaire de secrets de Colab (icône clé) ;
- **ne jamais partager ce carnet une fois exécuté** : ses sorties contiennent le contenu des dossiers.

## 1. Installation, Drive, clés

In [ ]:
%pip install -q --upgrade --force-reinstall --no-deps "hazid @ git+https://github.com/MoustaphaRouf/llm-hazard-id-eval"
%pip install -q anthropic openai google-genai openpyxl scikit-learn
from google.colab import drive, userdata
import os, pathlib
drive.mount('/content/drive')

BASE_DIR = pathlib.Path('/content/drive/MyDrive/ACADEMIE/AI')   # dossier contenant le canevas
for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']:
    try:
        os.environ[k] = userdata.get(k); print('OK      ', k)
    except Exception:
        print('ABSENTE ', k)

## 2. Canevas et choix des dossiers du pilote

In [ ]:
import pandas as pd
from hazid import adjudication, canvas, contamination, generation, matching
from hazid.protocol import Protocol

CANEVAS = BASE_DIR / 'Synthese-eds_V5_anonymise.xlsx'      # canevas ANONYMISÉ uniquement

# ÉCART AU PROTOCOLE (à consigner au journal) : pilote sur le libellé verbatim du panel,
# la reformulation normalisée n'étant pas renseignée. Les modèles rédigent sous la forme
# « événement conduisant à conséquence » : l'appariement lexical sera plus sévère pour eux,
# d'où l'importance du contrôle manuel des appariements au §7.
COLONNE_DANGERS = 'Libellé du danger (verbatim)'
dossiers = canvas.load_canvas(CANEVAS, hazard_column=COLONNE_DANGERS)
etat = pd.DataFrame([{'Dossier': k, 'Contexte (car.)': len(d.context), 'Dangers panel': len(d.human_hazards)}
                     for k, d in sorted(dossiers.items())])
display(etat)

complets = [k for k, d in sorted(dossiers.items()) if len(d.human_hazards) >= 3 and len(d.context) >= 300]
assert len(complets) >= 3, (f"Seulement {len(complets)} dossier(s) complet(s) : renseigner au moins 3 dossiers "
                            "(contexte pré-séance + au moins 3 dangers du panel) dans le canevas.")
# Dossiers du pilote : aucun danger « Ajout Claude » dans la référence, 8 à 18 dangers chacun.
CALIBRAGE = 'EDS-04'           # sert uniquement à fixer le seuil
MESURE = ['EDS-02', 'EDS-09']  # dossiers sur lesquels porteront les résultats
assert all(k in complets for k in [CALIBRAGE, *MESURE]), 'Dossier du pilote incomplet dans le canevas.'
pilote = {k: dossiers[k] for k in [CALIBRAGE, *MESURE]}
print('Calibrage :', CALIBRAGE, '| Mesure :', MESURE)

## 3. Contrôle anti-contamination
Toute alerte doit être levée avant de poursuivre.

In [ ]:
alertes = contamination.check(pilote)
display(pd.DataFrame(alertes)) if alertes else print('Aucune contamination manifeste détectée.')
assert not alertes, 'Lever les alertes avant de poursuivre.'

## 4. Gel du protocole

Les identifiants doivent être des **versions exactes** (pas d'alias `latest`). Corriger `MODELES`
jusqu'à ce que la vérification soit entièrement OK, puis consigner l'empreinte au journal de l'étude.

In [ ]:
MODELES = {
    'A': {'provider': 'anthropic', 'model': 'claude-sonnet-4-6'},
    'B': {'provider': 'openai',    'model': 'gpt-5.1'},
    'C': {'provider': 'google',    'model': 'gemini-3.1-pro-preview'},  # version preview : à consigner au journal
}
protocole = Protocol(models=MODELES, temperature=1.0, n_runs=5)   # 1.0 pour tous : seule valeur acceptée par l'API Anthropic actuelle
ok = generation.check_models(protocole, list(MODELES))
assert ok, 'Corriger les identifiants de modèles.'
# NB : la vérification ne garantit pas que la génération soit ouverte ; le test à blanc ci-dessous essaie chaque modèle.
print("Empreinte du protocole (au journal) :", protocole.fingerprint())

## 5. Test à blanc — un dossier, un modèle, une exécution

In [ ]:
import shutil
shutil.rmtree(BASE_DIR / 'pilote_test_blanc', ignore_errors=True)   # repartir d'un test vierge
essai = Protocol(models=MODELES, temperature=1.0, n_runs=1)
generation.run_campaign({CALIBRAGE: pilote[CALIBRAGE]}, essai, BASE_DIR / 'pilote_test_blanc',
                        conditions=('blind',), keys=list(MODELES))
essais = generation.load_runs(BASE_DIR / 'pilote_test_blanc', ok_only=False)
for r in essais:
    print(f"{r['model_key']} ({r['model']}) : {r['status']}, {r['n_hazards']} dangers", (r['error'] or '')[:200])
assert all(r['status'] == 'ok' for r in essais), 'Corriger avant la campagne (crédit, identifiant de modèle...).'

## 6. Campagne pilote (90 appels, reprenable)

In [ ]:
RUNS = BASE_DIR / 'pilote_executions'
compte = generation.run_campaign(pilote, protocole, RUNS)
print(compte)
runs = generation.load_runs(RUNS)
print(len(runs), 'exécutions valides')

## 7. Appariement manuel assisté

Avec le libellé verbatim du panel, l'appariement par mots ne reconnaît pas deux formulations
différentes du même danger. L'appariement est donc fait **à la main**. Pour chaque danger du panel,
un modèle de similarité sémantique, qui tourne **dans cette session** (rien n'est envoyé), classe
toutes les propositions des modèles et n'en garde que les `K` plus proches (`K = 20` : au pilote, avec 10, des correspondances apparaissaient encore aux rangs 9 et 10). Le classement ne fait
que trier : la décision « même danger ? » t'appartient. Le modèle qui a fait chaque proposition
n'est **pas** indiqué, pour ne pas influencer le jugement.

Limite à consigner : une correspondance classée au-delà de `K` n'est jamais montrée, donc le
rappel obtenu est un **minorant**.

Le pilote a aussi montré que la similarité (par mots ou sémantique) ne remplace pas le jugement :
face au réviseur, AUC de 0,63 et précision d'au plus 13 % à tout seuil.

In [ ]:
%pip install -q sentence-transformers
from hazid import manual
K = 20
humains_complets = {k: d.human_hazards for k, d in pilote.items()}
for d in [CALIBRAGE, *MESURE]:
    t = manual.candidates(runs, humains_complets, d, k=K)
    manual.write_review_sheet(t, BASE_DIR / f'appariement_{d}.xlsx')
    print(f"{d} : {len(t)} paires -> appariement_{d}.xlsx (classement : {t.attrs['method']})")

**À faire maintenant, hors carnet :** ouvre chaque fichier `appariement_EDS-xx.xlsx` dans ton
Drive et, pour chaque ligne, choisis dans « Même danger ? » :

| Code | Règle |
|---|---|
| **Oui** | même danger |
| **Composante** | la proposition isole un élément d'un danger composite du rapport (ex. « vent, QNH et QFE erronés » → « vent erroné » et « QNH/QFE erronés ») |
| **Recouvrement** | autre recoupement partiel : plus général, conséquence différente… |
| **Non** | danger différent |

Enregistre au format .xlsx, puis exécute la cellule suivante. Pour mesurer l'accord inter-codeurs,
fais recoder un dossier par une seconde personne qui applique cette règle écrite.

## 8. Résultats

In [ ]:
import pandas as pd
revues = pd.concat([pd.read_excel(BASE_DIR / f'appariement_{d}.xlsx', dtype={'Même danger ?': object})
                    for d in [CALIBRAGE, *MESURE]])
variantes = {'strict': False, 'composante': 'component', 'large': True}   # « Partiel » (pilote) compte en large
for nom, crit in variantes.items():
    r = manual.recall_from_review(revues, runs, count_partial=crit)
    r.to_csv(BASE_DIR / f'pilote_rappel_{nom}.csv', index=False)
print('Paires sans réponse :', r.attrs['unanswered'])
display(manual.recall_from_review(revues, runs, count_partial='component'))

# Trois modèles combinés, condition aveugle
aveugle = [dict(r, model_key='A+B+C') for r in runs if r['condition'] == 'blind']
display(manual.recall_from_review(revues, aveugle, count_partial='component'))

### Stabilité et accord entre modèles — à interpréter avec prudence

Ces deux indicateurs reposent sur l'appariement **par mots** entre formulations des modèles. Au pilote,
ils étaient proches de zéro, ce qui est très probablement un artefact de mesure : ne pas les citer sans
les avoir vérifiés sur un échantillon jugé à la main.

In [ ]:
SEUIL_MODELES = 0.55
stab = matching.metrics(runs, {k: [] for k in pilote}, SEUIL_MODELES)[
    ['dossier', 'model', 'condition', 'hazards_mean', 'count_sd', 'content_stability']]
display(stab); stab.to_csv(BASE_DIR / 'pilote_stabilite.csv', index=False)
accord = matching.agreement(runs, SEUIL_MODELES)
display(accord); accord.to_csv(BASE_DIR / 'pilote_accord.csv', index=False)

## 9. Classeur d'adjudication en aveugle (validité des dangers nouveaux)

Tous les dangers du panel sont gardés ; les dangers des modèles sont **échantillonnés** (50 par dossier,
graine fixe) pour que des pairs puissent juger en une heure environ — le taux de validité estimé sur
l'échantillon vaut pour l'ensemble. Adjudicateurs : des pairs qui ne connaissent pas les dossiers
(ni le responsable de l'étude, ni le stagiaire). Transmettre **uniquement**
`pilote-adjudication-a-transmettre.xlsx` ; garder la clé jusqu'à restitution complète des jugements.

In [ ]:
complet = BASE_DIR / 'pilote-adjudication-aveugle.xlsx'
transmis = BASE_DIR / 'pilote-adjudication-a-transmettre.xlsx'
evaluation = [r for r in runs if r['dossier'] in MESURE]
n = adjudication.write_workbook(evaluation, {k: pilote[k].human_hazards for k in MESURE},
                                complet, protocole.shuffle_seed, collapse_threshold=SEUIL_MODELES,
                                max_model_per_dossier=50)
adjudication.export_for_adjudicators(complet, transmis)
print(n, 'lignes à adjudiquer ->', transmis.name)